# Notebook 06: Network Spike Classification

## Project
NEM Network Constraints & Price Divergence Intelligence System

## Business Question
Can price spikes in NSW1 and VIC1 be classified by their likely market driver using demand, volatility, constraint, interconnector, and price divergence features?

## Purpose Of This Notebook
Notebook 06 converts price spike flags into market explanations.

In earlier notebooks, we built the analytical dataset step by step:

- Notebook 01 created the clean base market context table.
- Notebook 02 engineered base market features such as price spikes, supply-demand gap, price change, and volatility.
- Notebook 03 added constraint activity features.
- Notebook 04 added interconnector flow and congestion-style signals.
- Notebook 05 added NSW1-VIC1 price divergence features.

This notebook uses all of those features to classify price spikes into likely explanatory categories.


## How This Notebook Links To The Full Project

The goal of Project 2 is not only to detect price spikes. The goal is to explain them in a way that is useful for energy traders, analysts, and modellers.

A price spike can occur for different reasons:

- local supply-demand tightness
- high volatility
- network constraint activity
- interconnector or congestion-style behaviour
- regional price separation
- unclear or mixed conditions

Notebook 06 creates a structured classification layer so that later decision intelligence and dashboard outputs can focus on why events happened, not only when they happened.


## Spike Classification Logic

This notebook classifies spike intervals into the following categories:

| Classification | Meaning | Indicative Signals |
|---|---|---|
| `Constraint-driven` | Spike occurred during elevated constraint activity or strong marginal value signals. | High active constraint count, high constraint marginal value, violation signal |
| `Interconnector/congestion-driven` | Spike occurred during interconnector stress or regional price separation. | Congestion-style signal, high flow change, divergence flag, extreme divergence flag |
| `Demand-driven` | Spike occurred during tighter regional supply-demand conditions. | Low supply-demand gap, high demand relative to region history |
| `Volatility-driven` | Spike occurred during unstable price conditions. | High rolling price volatility, large price change |
| `Unknown` | Spike does not strongly match the above signals. | No clear dominant signal |

### Important Analytical Note
This is a rule-based classification, not a causal model. The classification should be treated as decision-support evidence, not proof of causality.


## Inputs And Outputs

### Inputs
This notebook uses:

- `outputs/05_market_features_with_divergence.csv`

### Outputs
This notebook will create:

- `outputs/06_network_spike_explanation.csv`
- `outputs/06_spike_classification_summary.csv`
- `outputs/06_market_features_with_spike_classification.csv`

The final output will be used in the next notebook to generate decision intelligence recommendations.


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 140)


In [2]:
PROJECT_ROOT = (
    Path.cwd().parents[0]
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"

MARKET_DIVERGENCE_FILE = OUTPUT_DIR / "05_market_features_with_divergence.csv"

print("Project root:", PROJECT_ROOT)
print("Input file:", MARKET_DIVERGENCE_FILE)
print("Output directory:", OUTPUT_DIR)


Project root: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence
Input file: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/05_market_features_with_divergence.csv
Output directory: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs


## Step 3: Load Market Features With Divergence

### What we are doing
We are loading the full feature table created in Notebook 05.

### Why it matters
This dataset contains all signals required for spike classification: price, demand, volatility, constraints, interconnectors, and NSW1-VIC1 price divergence.


In [3]:
market_classification = pd.read_csv(
    MARKET_DIVERGENCE_FILE,
    parse_dates=["settlementdate"]
)

print("Market classification shape:", market_classification.shape)

market_classification.head()


Market classification shape: (16126, 43)


,settlementdate,regionid,rrp,intervention,totaldemand,availablegeneration,netinterchange,intervention_regionsum,supply_demand_gap,price_spike_flag,price_change,rolling_rrp_volatility_1h,date,hour,weekday,is_weekend,active_constraint_count,max_constraint_marginal_value,total_constraint_marginal_value,violation_flag,max_violation_degree,total_absolute_flow,max_absolute_flow,average_absolute_flow,max_absolute_flow_change,high_flow_interconnector_count,high_flow_change_interconnector_count,congestion_signal_count,congestion_signal_flag,vic_nsw_flow,vic_nsw_absolute_flow,vic_nsw_flow_change,vic_nsw_absolute_flow_change,vic_nsw_high_flow_flag,vic_nsw_high_flow_change_flag,vic_nsw_congestion_signal_flag,nsw_rrp,vic_rrp,price_spread,absolute_price_spread,divergence_flag,extreme_divergence_flag,divergence_direction
0,2026-02-01 00:05:00,NSW1,57.05984,0,8186.33,13272.53373,-1054.73,0,5086.20373,False,NaN,NaN,2026-02-01,0,Sunday,True,17,135.3015,-902.14975,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,57.05984,47.06047,9.99937,9.99937,False,False,NSW1 higher than VIC1
1,2026-02-01 00:10:00,NSW1,64.51979,0,8165.29,13246.75420,-924.67,0,5081.46420,False,7.45995,NaN,2026-02-01,0,Sunday,True,16,40.2794,-1006.87256,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,64.51979,54.34465,10.17514,10.17514,False,False,NSW1 higher than VIC1
2,2026-02-01 00:15:00,NSW1,65.08727,0,8202.70,13198.51267,-1006.50,0,4995.81267,False,0.56748,4.479816,2026-02-01,0,Sunday,True,20,9.1600,-1053.75439,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,65.08727,19.18000,45.90727,45.90727,False,False,NSW1 higher than VIC1
3,2026-02-01 00:20:00,NSW1,64.89000,0,8213.41,13169.71135,-1011.73,0,4956.30135,False,-0.19727,3.893369,2026-02-01,0,Sunday,True,20,4.3600,-1061.42631,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,64.89000,19.18000,45.71000,45.71000,False,False,NSW1 higher than VIC1
4,2026-02-01 00:25:00,NSW1,63.47180,0,8055.53,13139.84760,-934.04,0,5084.31760,False,-1.41820,3.381808,2026-02-01,0,Sunday,True,20,4.3600,-1060.65174,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,63.47180,15.18326,48.28854,48.28854,False,False,NSW1 higher than VIC1


## Step 4: Confirm Spike Intervals

### What we are doing
We are checking how many price spike intervals exist by region.

### Why it matters
Only intervals with `price_spike_flag = True` will receive a spike classification. Non-spike intervals remain useful context but are not classified as spike events.


In [4]:
spike_count_check = (
    market_classification
    .groupby("regionid")
    .agg(
        intervals=("settlementdate", "count"),
        spike_intervals=("price_spike_flag", "sum"),
        max_rrp=("rrp", "max"),
        average_rrp=("rrp", "mean"),
    )
    .reset_index()
)

spike_count_check


,regionid,intervals,spike_intervals,max_rrp,average_rrp
0,NSW1,8063,53,20300.00000,83.762187
1,VIC1,8063,0,269.77601,45.055455


## Step 5: Check Classification Feature Availability

### What we are doing
We are checking that the required classification features are present in the dataset.

### Why it matters
The rule-based classifier depends on features created across Notebooks 02-05. This check prevents silent errors before classification.


In [5]:
required_classification_features = [
    "settlementdate",
    "regionid",
    "rrp",
    "price_spike_flag",
    "totaldemand",
    "availablegeneration",
    "supply_demand_gap",
    "price_change",
    "rolling_rrp_volatility_1h",
    "active_constraint_count",
    "max_constraint_marginal_value",
    "violation_flag",
    "congestion_signal_flag",
    "congestion_signal_count",
    "vic_nsw_congestion_signal_flag",
    "divergence_flag",
    "extreme_divergence_flag",
    "absolute_price_spread",
]

missing_features = [
    col for col in required_classification_features
    if col not in market_classification.columns
]

if missing_features:
    print("Missing features:", missing_features)
else:
    print("All required classification features are available.")


All required classification features are available.


## Step 6: Create Classification Thresholds

### What we are doing
We are creating data-driven thresholds for demand tightness, volatility, price change, and constraint activity.

### Why it matters
Different regions have different normal operating ranges. Using percentile-based thresholds allows the classification rules to adapt to the observed February 2026 dataset instead of relying only on fixed assumptions.


In [6]:
thresholds_by_region = (
    market_classification
    .groupby("regionid")
    .agg(
        high_demand_threshold=("totaldemand", lambda x: x.quantile(0.90)),
        low_supply_demand_gap_threshold=("supply_demand_gap", lambda x: x.quantile(0.10)),
        high_volatility_threshold=("rolling_rrp_volatility_1h", lambda x: x.quantile(0.90)),
        high_price_change_threshold=("price_change", lambda x: x.abs().quantile(0.90)),
        high_constraint_count_threshold=("active_constraint_count", lambda x: x.quantile(0.90)),
        high_constraint_mv_threshold=("max_constraint_marginal_value", lambda x: x.quantile(0.90)),
    )
    .reset_index()
)

thresholds_by_region


,regionid,high_demand_threshold,low_supply_demand_gap_threshold,high_volatility_threshold,high_price_change_threshold,high_constraint_count_threshold,high_constraint_mv_threshold
0,NSW1,9834.070,3891.103892,20.448633,20.990000,25.0,15.37
1,VIC1,5896.188,3715.499284,19.836386,18.827142,25.0,15.37


## Step 7: Join Thresholds To Market Classification Table

### What we are doing
We are joining the region-specific thresholds back to the interval-level dataset.

### Why it matters
Each interval can now be tested against the correct threshold for its region.


In [7]:
market_classification = market_classification.merge(
    thresholds_by_region,
    on="regionid",
    how="left"
)

market_classification.head()


,settlementdate,regionid,rrp,intervention,totaldemand,availablegeneration,netinterchange,intervention_regionsum,supply_demand_gap,price_spike_flag,price_change,rolling_rrp_volatility_1h,date,hour,weekday,is_weekend,active_constraint_count,max_constraint_marginal_value,total_constraint_marginal_value,violation_flag,max_violation_degree,total_absolute_flow,max_absolute_flow,average_absolute_flow,max_absolute_flow_change,high_flow_interconnector_count,high_flow_change_interconnector_count,congestion_signal_count,congestion_signal_flag,vic_nsw_flow,vic_nsw_absolute_flow,vic_nsw_flow_change,vic_nsw_absolute_flow_change,vic_nsw_high_flow_flag,vic_nsw_high_flow_change_flag,vic_nsw_congestion_signal_flag,nsw_rrp,vic_rrp,price_spread,absolute_price_spread,divergence_flag,extreme_divergence_flag,divergence_direction,high_demand_threshold,low_supply_demand_gap_threshold,high_volatility_threshold,high_price_change_threshold,high_constraint_count_threshold,high_constraint_mv_threshold
0,2026-02-01 00:05:00,NSW1,57.05984,0,8186.33,13272.53373,-1054.73,0,5086.20373,False,NaN,NaN,2026-02-01,0,Sunday,True,17,135.3015,-902.14975,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,57.05984,47.06047,9.99937,9.99937,False,False,NSW1 higher than VIC1,9834.07,3891.103892,20.448633,20.99,25.0,15.37
1,2026-02-01 00:10:00,NSW1,64.51979,0,8165.29,13246.75420,-924.67,0,5081.46420,False,7.45995,NaN,2026-02-01,0,Sunday,True,16,40.2794,-1006.87256,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,64.51979,54.34465,10.17514,10.17514,False,False,NSW1 higher than VIC1,9834.07,3891.103892,20.448633,20.99,25.0,15.37
2,2026-02-01 00:15:00,NSW1,65.08727,0,8202.70,13198.51267,-1006.50,0,4995.81267,False,0.56748,4.479816,2026-02-01,0,Sunday,True,20,9.1600,-1053.75439,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,65.08727,19.18000,45.90727,45.90727,False,False,NSW1 higher than VIC1,9834.07,3891.103892,20.448633,20.99,25.0,15.37
3,2026-02-01 00:20:00,NSW1,64.89000,0,8213.41,13169.71135,-1011.73,0,4956.30135,False,-0.19727,3.893369,2026-02-01,0,Sunday,True,20,4.3600,-1061.42631,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,64.89000,19.18000,45.71000,45.71000,False,False,NSW1 higher than VIC1,9834.07,3891.103892,20.448633,20.99,25.0,15.37
4,2026-02-01 00:25:00,NSW1,63.47180,0,8055.53,13139.84760,-934.04,0,5084.31760,False,-1.41820,3.381808,2026-02-01,0,Sunday,True,20,4.3600,-1060.65174,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,63.47180,15.18326,48.28854,48.28854,False,False,NSW1 higher than VIC1,9834.07,3891.103892,20.448633,20.99,25.0,15.37


## Step 8: Create Driver Signal Flags

### What we are doing
We are creating boolean flags that indicate whether each interval shows demand, volatility, constraint, or interconnector/divergence stress.

### Why it matters
The final spike classification will be based on these driver signal flags.


In [8]:
market_classification["demand_tightness_signal"] = (
    (market_classification["totaldemand"] >= market_classification["high_demand_threshold"])
    | (market_classification["supply_demand_gap"] <= market_classification["low_supply_demand_gap_threshold"])
)

market_classification["volatility_signal"] = (
    (market_classification["rolling_rrp_volatility_1h"] >= market_classification["high_volatility_threshold"])
    | (market_classification["price_change"].abs() >= market_classification["high_price_change_threshold"])
)

market_classification["constraint_signal"] = (
    (market_classification["active_constraint_count"] >= market_classification["high_constraint_count_threshold"])
    | (market_classification["max_constraint_marginal_value"] >= market_classification["high_constraint_mv_threshold"])
    | (market_classification["violation_flag"] == True)
)

market_classification["interconnector_congestion_signal"] = (
    (market_classification["congestion_signal_flag"] == True)
    | (market_classification["congestion_signal_count"] >= 2)
    | (market_classification["vic_nsw_congestion_signal_flag"] == True)
    | (market_classification["divergence_flag"] == True)
    | (market_classification["extreme_divergence_flag"] == True)
)

market_classification[
    [
        "settlementdate",
        "regionid",
        "rrp",
        "price_spike_flag",
        "demand_tightness_signal",
        "volatility_signal",
        "constraint_signal",
        "interconnector_congestion_signal",
    ]
].head()


,settlementdate,regionid,rrp,price_spike_flag,demand_tightness_signal,volatility_signal,constraint_signal,interconnector_congestion_signal
0,2026-02-01 00:05:00,NSW1,57.05984,False,False,False,True,False
1,2026-02-01 00:10:00,NSW1,64.51979,False,False,False,True,False
2,2026-02-01 00:15:00,NSW1,65.08727,False,False,False,False,False
3,2026-02-01 00:20:00,NSW1,64.89000,False,False,False,False,False
4,2026-02-01 00:25:00,NSW1,63.47180,False,False,False,False,False


## Step 9: Validate Driver Signals During Spike Intervals

### What we are doing
We are summarising how often each driver signal appears during price spike intervals.

### Why it matters
This gives a first view of which conditions are most common during high-price events before assigning a single classification label.


In [9]:
spike_signal_summary = (
    market_classification[market_classification["price_spike_flag"]]
    .groupby("regionid")
    .agg(
        spike_intervals=("settlementdate", "count"),
        demand_signal_intervals=("demand_tightness_signal", "sum"),
        volatility_signal_intervals=("volatility_signal", "sum"),
        constraint_signal_intervals=("constraint_signal", "sum"),
        interconnector_signal_intervals=("interconnector_congestion_signal", "sum"),
    )
    .reset_index()
)

spike_signal_summary


,regionid,spike_intervals,demand_signal_intervals,volatility_signal_intervals,constraint_signal_intervals,interconnector_signal_intervals
0,NSW1,53,49,53,27,53


## Step 10: Define Spike Classification Rule

### What we are doing
We are defining a rule-based function that assigns each spike interval to one likely driver category.

### Why it matters
This converts multiple driver signals into a simple classification that can be used in reporting, visualisation, and decision intelligence.

### Priority order
1. Constraint-driven
2. Interconnector/congestion-driven
3. Demand-driven
4. Volatility-driven
5. Unknown

The order is intentional. If a spike has a strong network signal, we want to capture that before assigning it to demand or volatility.


In [10]:
def classify_spike_driver(row):
    if not row["price_spike_flag"]:
        return "Not a spike"

    if row["constraint_signal"]:
        return "Constraint-driven"

    if row["interconnector_congestion_signal"]:
        return "Interconnector/congestion-driven"

    if row["demand_tightness_signal"]:
        return "Demand-driven"

    if row["volatility_signal"]:
        return "Volatility-driven"

    return "Unknown"


market_classification["spike_driver_classification"] = (
    market_classification.apply(classify_spike_driver, axis=1)
)

market_classification[
    [
        "settlementdate",
        "regionid",
        "rrp",
        "price_spike_flag",
        "spike_driver_classification",
    ]
].head()


,settlementdate,regionid,rrp,price_spike_flag,spike_driver_classification
0,2026-02-01 00:05:00,NSW1,57.05984,False,Not a spike
1,2026-02-01 00:10:00,NSW1,64.51979,False,Not a spike
2,2026-02-01 00:15:00,NSW1,65.08727,False,Not a spike
3,2026-02-01 00:20:00,NSW1,64.89000,False,Not a spike
4,2026-02-01 00:25:00,NSW1,63.47180,False,Not a spike


## Step 11: Review Spike Classification Results

### What we are doing
We are filtering to price spike intervals and reviewing their assigned driver classifications.

### Why it matters
This lets us check whether the rule-based classification produces reasonable explanations for the highest-price events.


In [12]:
network_spike_explanation = (
    market_classification[
        market_classification["price_spike_flag"]
    ]
    [
        [
            "settlementdate",
            "regionid",
            "rrp",
            "spike_driver_classification",
            "demand_tightness_signal",
            "volatility_signal",
            "constraint_signal",
            "interconnector_congestion_signal",
            "totaldemand",
            "availablegeneration",
            "supply_demand_gap",
            "price_change",
            "rolling_rrp_volatility_1h",
            "active_constraint_count",
            "max_constraint_marginal_value",
            "congestion_signal_count",
            "vic_nsw_congestion_signal_flag",
            "absolute_price_spread",
            "divergence_flag",
            "extreme_divergence_flag",
        ]
    ]
    .sort_values("rrp", ascending=False)
    .reset_index(drop=True)
)

network_spike_explanation.head(30)


,settlementdate,regionid,rrp,spike_driver_classification,demand_tightness_signal,volatility_signal,constraint_signal,interconnector_congestion_signal,totaldemand,availablegeneration,supply_demand_gap,price_change,rolling_rrp_volatility_1h,active_constraint_count,max_constraint_marginal_value,congestion_signal_count,vic_nsw_congestion_signal_flag,absolute_price_spread,divergence_flag,extreme_divergence_flag
0,2026-02-05 14:30:00,NSW1,20300.00000,Interconnector/congestion-driven,True,True,False,True,10190.15,15492.52355,5302.37355,19842.65742,5793.809073,24,12.12000,4.0,True,20308.00000,True,True
1,2026-02-06 14:05:00,NSW1,19036.53000,Interconnector/congestion-driven,False,True,False,True,9651.93,15075.06206,5423.13206,18882.33000,5801.771238,23,5.34000,2.0,False,19022.81000,True,True
2,2026-02-05 18:05:00,NSW1,9914.02372,Constraint-driven,True,True,True,True,11811.76,13709.77609,1898.01609,9634.29090,3719.949900,22,20.23000,2.0,False,9794.69790,True,True
3,2026-02-05 17:55:00,NSW1,9911.42681,Interconnector/congestion-driven,True,True,False,True,11901.45,14019.02209,2117.57209,9608.77893,2762.209380,24,4.96000,2.0,False,9854.92681,True,True
4,2026-02-05 16:35:00,NSW1,9240.83000,Constraint-driven,True,True,True,True,11896.59,14384.92803,2488.33803,8641.47000,2519.281003,29,4.34000,2.0,False,9231.88000,True,True
5,2026-02-07 16:40:00,NSW1,9199.30000,Interconnector/congestion-driven,True,True,False,True,11073.40,13443.29640,2369.89640,9092.46000,2617.222174,21,10.41000,2.0,False,9142.80000,True,True
6,2026-02-05 14:50:00,NSW1,9199.30000,Interconnector/congestion-driven,True,True,False,True,10400.88,15793.28831,5392.40831,8801.87165,6180.726188,23,7.45000,3.0,True,9206.06000,True,True
7,2026-02-06 13:35:00,NSW1,9199.30000,Constraint-driven,False,True,True,True,9613.17,15758.70193,6145.53193,8899.55000,2580.566888,26,7.99000,4.0,True,9206.00383,True,True
8,2026-02-05 14:40:00,NSW1,7380.80815,Interconnector/congestion-driven,True,True,False,True,10203.66,15819.78989,5616.12989,6908.09033,5953.998909,24,9.19000,2.0,True,7388.80815,True,True
9,2026-02-05 18:15:00,NSW1,1278.88000,Interconnector/congestion-driven,True,True,False,True,11744.07,13349.01575,1604.94575,1012.34851,3695.194359,21,3.96000,2.0,False,1178.88000,True,True


## Step 12: Summarise Spike Classifications

### What we are doing
We are counting how many spike intervals were assigned to each driver category.

### Why it matters
This shows the dominant explanation pattern for price spikes in the study period.


In [13]:
spike_classification_summary = (
    network_spike_explanation
    .groupby(["regionid", "spike_driver_classification"])
    .agg(
        spike_intervals=("settlementdate", "count"),
        average_rrp=("rrp", "mean"),
        max_rrp=("rrp", "max"),
        average_supply_demand_gap=("supply_demand_gap", "mean"),
        average_active_constraints=("active_constraint_count", "mean"),
        average_constraint_marginal_value=("max_constraint_marginal_value", "mean"),
        average_congestion_signal_count=("congestion_signal_count", "mean"),
        average_absolute_price_spread=("absolute_price_spread", "mean"),
    )
    .reset_index()
    .sort_values(["regionid", "spike_intervals"], ascending=[True, False])
)

spike_classification_summary


,regionid,spike_driver_classification,spike_intervals,average_rrp,max_rrp,average_supply_demand_gap,average_active_constraints,average_constraint_marginal_value,average_congestion_signal_count,average_absolute_price_spread
0,NSW1,Constraint-driven,27,1522.014782,9914.02372,2900.400581,26.592593,11.392472,1.703704,1478.041334
1,NSW1,Interconnector/congestion-driven,26,3346.608450,20300.00000,3675.721398,22.153846,7.022439,2.192308,3323.216936


## Step 13: Create Classification Confidence

### What we are doing
We are assigning a simple confidence level based on how many driver signals are present.

### Why it matters
Some spike classifications are clearer than others. A spike with multiple supporting signals should be treated with higher confidence than a spike with only one weak signal.


In [14]:
driver_signal_cols = [
    "demand_tightness_signal",
    "volatility_signal",
    "constraint_signal",
    "interconnector_congestion_signal",
]

market_classification["driver_signal_count"] = (
    market_classification[driver_signal_cols]
    .sum(axis=1)
)

def assign_classification_confidence(row):
    if not row["price_spike_flag"]:
        return "Not applicable"

    if row["spike_driver_classification"] == "Unknown":
        return "Low"

    if row["driver_signal_count"] >= 2:
        return "High"

    return "Medium"


market_classification["classification_confidence"] = (
    market_classification.apply(assign_classification_confidence, axis=1)
)

market_classification[
    [
        "settlementdate",
        "regionid",
        "rrp",
        "spike_driver_classification",
        "driver_signal_count",
        "classification_confidence",
    ]
].head()


,settlementdate,regionid,rrp,spike_driver_classification,driver_signal_count,classification_confidence
0,2026-02-01 00:05:00,NSW1,57.05984,Not a spike,1,Not applicable
1,2026-02-01 00:10:00,NSW1,64.51979,Not a spike,1,Not applicable
2,2026-02-01 00:15:00,NSW1,65.08727,Not a spike,0,Not applicable
3,2026-02-01 00:20:00,NSW1,64.89000,Not a spike,0,Not applicable
4,2026-02-01 00:25:00,NSW1,63.47180,Not a spike,0,Not applicable


## Step 14: Add Classification Confidence To Spike Explanation

### What we are doing
We are updating the spike explanation table with driver signal counts and confidence levels.

### Why it matters
The final spike explanation output should include both classification and confidence so users understand how strongly the rule-based logic supports each label.


In [15]:
network_spike_explanation = (
    market_classification[
        market_classification["price_spike_flag"]
    ]
    [
        [
            "settlementdate",
            "regionid",
            "rrp",
            "spike_driver_classification",
            "classification_confidence",
            "driver_signal_count",
            "demand_tightness_signal",
            "volatility_signal",
            "constraint_signal",
            "interconnector_congestion_signal",
            "totaldemand",
            "availablegeneration",
            "supply_demand_gap",
            "price_change",
            "rolling_rrp_volatility_1h",
            "active_constraint_count",
            "max_constraint_marginal_value",
            "congestion_signal_count",
            "vic_nsw_congestion_signal_flag",
            "absolute_price_spread",
            "divergence_flag",
            "extreme_divergence_flag",
        ]
    ]
    .sort_values("rrp", ascending=False)
    .reset_index(drop=True)
)

network_spike_explanation.head(30)


,settlementdate,regionid,rrp,spike_driver_classification,classification_confidence,driver_signal_count,demand_tightness_signal,volatility_signal,constraint_signal,interconnector_congestion_signal,totaldemand,availablegeneration,supply_demand_gap,price_change,rolling_rrp_volatility_1h,active_constraint_count,max_constraint_marginal_value,congestion_signal_count,vic_nsw_congestion_signal_flag,absolute_price_spread,divergence_flag,extreme_divergence_flag
0,2026-02-05 14:30:00,NSW1,20300.00000,Interconnector/congestion-driven,High,3,True,True,False,True,10190.15,15492.52355,5302.37355,19842.65742,5793.809073,24,12.12000,4.0,True,20308.00000,True,True
1,2026-02-06 14:05:00,NSW1,19036.53000,Interconnector/congestion-driven,High,2,False,True,False,True,9651.93,15075.06206,5423.13206,18882.33000,5801.771238,23,5.34000,2.0,False,19022.81000,True,True
2,2026-02-05 18:05:00,NSW1,9914.02372,Constraint-driven,High,4,True,True,True,True,11811.76,13709.77609,1898.01609,9634.29090,3719.949900,22,20.23000,2.0,False,9794.69790,True,True
3,2026-02-05 17:55:00,NSW1,9911.42681,Interconnector/congestion-driven,High,3,True,True,False,True,11901.45,14019.02209,2117.57209,9608.77893,2762.209380,24,4.96000,2.0,False,9854.92681,True,True
4,2026-02-05 16:35:00,NSW1,9240.83000,Constraint-driven,High,4,True,True,True,True,11896.59,14384.92803,2488.33803,8641.47000,2519.281003,29,4.34000,2.0,False,9231.88000,True,True
5,2026-02-07 16:40:00,NSW1,9199.30000,Interconnector/congestion-driven,High,3,True,True,False,True,11073.40,13443.29640,2369.89640,9092.46000,2617.222174,21,10.41000,2.0,False,9142.80000,True,True
6,2026-02-05 14:50:00,NSW1,9199.30000,Interconnector/congestion-driven,High,3,True,True,False,True,10400.88,15793.28831,5392.40831,8801.87165,6180.726188,23,7.45000,3.0,True,9206.06000,True,True
7,2026-02-06 13:35:00,NSW1,9199.30000,Constraint-driven,High,3,False,True,True,True,9613.17,15758.70193,6145.53193,8899.55000,2580.566888,26,7.99000,4.0,True,9206.00383,True,True
8,2026-02-05 14:40:00,NSW1,7380.80815,Interconnector/congestion-driven,High,3,True,True,False,True,10203.66,15819.78989,5616.12989,6908.09033,5953.998909,24,9.19000,2.0,True,7388.80815,True,True
9,2026-02-05 18:15:00,NSW1,1278.88000,Interconnector/congestion-driven,High,3,True,True,False,True,11744.07,13349.01575,1604.94575,1012.34851,3695.194359,21,3.96000,2.0,False,1178.88000,True,True


## Step 15: Save Notebook 06 Outputs

### What we are doing
We are saving the spike explanation table, classification summary, and full dataset with spike classification fields.

### Why it matters
These outputs are used in the next notebook to generate decision intelligence recommendations.


In [17]:
network_spike_explanation_output = OUTPUT_DIR / "06_network_spike_explanation.csv"
spike_classification_summary_output = OUTPUT_DIR / "06_spike_classification_summary.csv"
market_with_spike_classification_output = OUTPUT_DIR / "06_market_features_with_spike_classification.csv"

network_spike_explanation.to_csv(network_spike_explanation_output, index=False)
spike_classification_summary.to_csv(spike_classification_summary_output, index=False)
market_classification.to_csv(market_with_spike_classification_output, index=False)

print("Saved:", network_spike_explanation_output)
print("Saved:", spike_classification_summary_output)
print("Saved:", market_with_spike_classification_output)


Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/06_network_spike_explanation.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/06_spike_classification_summary.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/06_market_features_with_spike_classification.csv


## Notebook 06 Analyst Note

Notebook 06 converted price spike flags into rule-based market explanations.

The classification logic used demand tightness, rolling volatility, price changes, constraint activity, interconnector congestion-style signals, and NSW1-VIC1 price divergence features created in the previous notebooks.

Each spike interval was classified into one of the following categories:

- Constraint-driven
- Interconnector/congestion-driven
- Demand-driven
- Volatility-driven
- Unknown

This classification is not a causal model. It is a transparent decision-support framework that helps analysts identify the most likely driver based on available market and network signals.

The most important output is:

`outputs/06_network_spike_explanation.csv`

This file provides a spike-level explanation table that can be used for Power BI, visualisation, and decision intelligence.


## Notebook 06 Summary Table

| Step | What We Did | Why It Matters | Output / Feature Created |
|---|---|---|---|
| Step 1-3 | Loaded libraries, project paths, and the market-divergence dataset from Notebook 05. | Brought together all engineered features needed for spike classification. | `market_classification` |
| Step 4 | Confirmed price spike intervals by region. | Verified which regions had spikes requiring classification. | `spike_count_check` |
| Step 5 | Checked required feature availability. | Confirmed all market, constraint, interconnector, and divergence fields were available. | `missing_features` check |
| Step 6 | Created region-specific classification thresholds. | Allowed demand, volatility, and constraint signals to be evaluated relative to each region's own behaviour. | `thresholds_by_region` |
| Step 7 | Joined thresholds to the interval dataset. | Made each interval testable against its region-specific thresholds. | Threshold columns |
| Step 8 | Created driver signal flags. | Translated raw features into demand, volatility, constraint, and interconnector/congestion signals. | `demand_tightness_signal`, `volatility_signal`, `constraint_signal`, `interconnector_congestion_signal` |
| Step 9 | Summarised driver signals during spike intervals. | Showed which signals were present during high-price events. | `spike_signal_summary` |
| Step 10 | Defined rule-based spike classification logic. | Converted multiple signals into a single explanatory category for each spike. | `spike_driver_classification` |
| Step 11 | Reviewed classified spike intervals. | Created the main spike explanation table for event review. | `network_spike_explanation` |
| Step 12 | Summarised classifications by region and driver. | Identified dominant spike explanation patterns. | `spike_classification_summary` |
| Step 13 | Created confidence levels. | Added judgement about how strongly the available signals support each classification. | `classification_confidence` |
| Step 14 | Added confidence to spike explanation output. | Made the final spike event table more useful for analyst review and dashboarding. | Updated `network_spike_explanation` |
| Step 15 | Saved Notebook 06 outputs. | Created reusable files for decision intelligence, visualisation, and Power BI. | `06_network_spike_explanation.csv`, `06_spike_classification_summary.csv`, `06_market_features_with_spike_classification.csv` |

### Key Analyst Takeaway

Notebook 06 moves the project from detection to explanation. Instead of only saying that a price spike occurred, the analysis now assigns a likely driver based on transparent market and network rules.

This is the bridge between technical feature engineering and decision intelligence. The next notebook will translate these classifications into market situations, insights, recommendations, risks, and confidence levels.
